# Salesinsight:

## Pipeline de Análise e Visualização de Dados de Vendas

### Importando bibliotecas iniciais

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import re
import matplotlib.pyplot as plt 
import seaborn as sns 
import os 
import json

### Importando o dataset e visualização das 6 primeiras linhas

In [ ]:
df_bruto = pd.read_csv('./vendas.csv', sep=';')

# Verificando as 6 primeiras linhas do dataset `vendas.csv`

df_bruto.head(6)

### Inspeção Inicial do Dataset

In [ ]:

def inspecionar_dados(df_bruto): 
    """Exibe informações básicas do DataFrame.""" 
    print("\n=== INSPEÇÃO INICIAL DO DATASET ===") 
    print(f"Shape: {df_bruto.shape}") 
    print(f"\nColunas: {list(df_bruto.columns)}") 
    print(f"\nTipos de dados:\n{df_bruto.dtypes}") 
    print(f"\nValores nulos por coluna:\n{df_bruto.isnull().sum()}") 
    print(f"\nPrimeiros registros:\n{df_bruto.head()}") 
    print(f"\nEstatísticas descritivas:\n{df_bruto.describe()}") 

In [ ]:
dados_brutos = inspecionar_dados(df_bruto)

In [ ]:

# Após rápida avaliação do dataset bruto, notou-se a presença de `"DATA INVÁLIDA"` em `data_venda`

df_bruto[df_bruto['data_venda'] == 'DATA INVÁLIDA']

In [ ]:

total = len(df_bruto)
invalidas = (df_bruto['data_venda'] == 'DATA INVÁLIDA').sum()
percentual = (invalidas / total) * 100

print(f"Datas inválidas: {invalidas} de {total} ({percentual:.2f}%)")

<br>

<u>Análise inicial do Dataset:</u>

1. 200 registros, 8 colunas
2. 6 nulos em `quantidade` e 10 nulos em `preco_unitario` — serão tratados na limpeza
3. `data_venda` como `str` — datas inválidas serão identificadas e tratadas antes da conversão para `datetime`
4. `quantidade` como `float64` — pode ser convertido para `int` após tratamento dos nulos

**Ponto de atenção:** `data_venda` aparece como `str` e `isnull().sum()` retorna 0 para ela — isso significa que linhas com `"DATA INVÁLIDA"` não são detectadas como nulas, pois são strings válidas para o pandas.

<br>

### Limpeza e tratamento dos dados

In [ ]:
df = df_bruto.copy()

def limpar_dados(df): 
    """ 
    Limpa e trata o DataFrame de vendas. 
    Retorna o DataFrame limpo e um relatório de limpeza. 
    """ 

    n_inicial = len(df) 
    relatorio = {} 
 
    # 1. Remover espaços extras em colunas de texto 
    colunas_texto = df.select_dtypes(include=["object", "string"]).columns
    for col in colunas_texto: 
        df[col] = df[col].str.strip()

    # 2. Usar regex para limpar caracteres especiais em 'cliente'
    if "cliente" in df.columns:
        df["cliente"] = df["cliente"].apply(lambda x: re.sub(r"[^A-Za-z0-9_]", "", str(x)))
 
    # 3. Converter data e remover datas inválidas 
    df["data_venda"] = pd.to_datetime(df["data_venda"], errors="coerce") 
    n_datas_invalidas = df["data_venda"].isnull().sum() 
    df = df.dropna(subset=["data_venda"]) 
    relatorio["datas_invalidas_removidas"] = n_datas_invalidas 
 
    # 4. Remover linhas com quantidade ou preço nulos 
    n_antes = len(df) 
    df = df.dropna(subset=["quantidade", "preco_unitario"]) 
    relatorio["linhas_nulas_removidas"] = n_antes - len(df) 
 
    # 5. Garantir tipos numéricos corretos 
    df["quantidade"] = df["quantidade"].astype(int) 
    df["preco_unitario"] = df["preco_unitario"].astype(float) 
 
    n_final = len(df) 
    relatorio["registros_iniciais"] = n_inicial 
    relatorio["registros_finais"] = n_final 
    relatorio["registros_removidos_total"] = n_inicial - n_final
    relatorio["percentual_removido"] = round((n_inicial - n_final) / n_inicial * 100, 2) # Informação adicionada
 
    print("\n=== RELATÓRIO DE LIMPEZA ===") 
    for chave, valor in relatorio.items(): 
        print(f" {chave}: {valor}")

    return df, relatorio

In [ ]:
# Chamando a função para geração do relatório da limpeza

df_limpo, relatorio = limpar_dados(df_bruto)

df_limpo.head(6)

<br>

<u>Avaliação pós limpeza do dataset:</u>

1. O dataset foi reduzido de 200 para 180 linhas após a remoção de nulos e das strings `"DATA INVÁLIDA"`. A perda total foi de 10%.
2. A coluna `data_venda` foi convertida com sucesso para `datetime64` e `quantidade` para `int`.
3. Espaços extras foram removidos e a coluna `cliente` foi normalizada via regex, restando apenas caracteres alfanuméricos.
4. O dataset está livre de nulos e com a tipagem correta, totalmente pronto para a engenharia de atributos e modelagem preditiva.

<br>

### Criação de colunas derivadas com transformações

In [ ]:
def aplicar_transformacao(serie, funcao_transformacao):
    """
    Aplica uma função recebida como parâmetro a uma Series.
    Demonstra o uso de função de ordem superior.
    """
    return serie.apply(funcao_transformacao)

In [ ]:
def criar_colunas_derivadas(df_limpo):
    """Cria colunas calculadas e derivadas a partir do dataset limpo.""" 
 
    # Receita total por linha de venda 
    df_limpo["receita_total"] = df_limpo["quantidade"] * df_limpo["preco_unitario"] 
 
    # Extração de componentes de data 
    df_limpo["mes"] = df_limpo["data_venda"].dt.month 
    df_limpo["mes_nome"] = df_limpo["data_venda"].dt.strftime("%B")  # nome do mês 
    df_limpo["trimestre"] = df_limpo["data_venda"].dt.quarter.apply(lambda q: f"Q{q}") 
    df_limpo["ano"] = df_limpo["data_venda"].dt.year 
 
    # Classificação da receita por item com numpy.select (transformação condicional vetorizada) 
    condicoes = [ 
        df_limpo["receita_total"] < 500, 
        (df_limpo["receita_total"] >= 500) & (df_limpo["receita_total"] < 5000), 
        df_limpo["receita_total"] >= 5000 
        ] 
    classificacoes = ["Baixo Valor", "Médio Valor", "Alto Valor"] 
    df_limpo["faixa_receita_item"] = np.select(condicoes, classificacoes, default="Não Classificado")

    print("\n=== COLUNAS DERIVADAS CRIADAS ===") 
    print(df_limpo[["data_venda", "receita_total", "mes", "trimestre", "faixa_receita_item"]].head()) 
    return df_limpo

In [ ]:
# Chamando a função de colunas derivadas garantindo uma cópia limpa do dataframe

df_transformado = criar_colunas_derivadas(df_limpo.copy())

df_transformado.head(5)

<br>

<u>Engenharia de Recursos (Colunas Derivadas):</u>

1. `receita_total`: Criada a variável alvo calculada (`quantidade` × `preco_unitario`).
2. Decomposição Temporal: Extraídos `mes`, `mes_nome`, `trimestre` e `ano` a partir de `data_venda` para capturar padrões de sazonalidade.
3. Classificação do faturamento dos itens em três faixas de valor (*Baixo, Médio e Alto*) de forma vetorizada via `np.select`.

<br>

### Cálculo de métricas agregadas

In [ ]:
def calcular_metricas(df_limpo): 
    """Calcula e retorna métricas agregadas do dataset.""" 
    metricas = {} 
    # Receita por mês 
    por_mes = df_limpo.groupby("mes").agg( receita_total=("receita_total", "sum"), 
        quantidade=("quantidade", "sum"), 
        n_vendas=("id_venda", "count") 
    ).reset_index().sort_values("mes") 
    metricas["receita por mes"] = por_mes 
 
    # Top 5 produtos por receita 
    top_produtos = df_limpo.groupby("produto")["receita_total"].sum().sort_values(ascending=False).head(5).reset_index() 
    metricas["top 5 produtos por receita"] = top_produtos 
 
    # Receita por categoria 
    por_categoria = df_limpo.groupby("categoria")["receita_total"].sum().reset_index() 
    metricas["receita por categoria"] = por_categoria 
 
    # Receita por região 
    por_regiao = df_limpo.groupby("regiao").agg( 
        receita_total=("receita_total", "sum"), 
        media_ticket=("receita_total", "mean") 
    ).reset_index().sort_values("receita_total", ascending=False) 
    metricas["receita por regiao"] = por_regiao 
 
    # Exibição 
    for nome, tabela in metricas.items(): 
        print(f"\n=== {nome.upper().replace('_', ' ')} ===") 
        print(tabela.to_string(index=False)) 
 
    return metricas

In [ ]:
# Chamando a função calcular_métricas
metricas_finais = calcular_metricas(df_transformado)

<u>Cálculo de Métricas Agregadas:</u>

1. Sazonalidade Mensal: o mês de maior faturamento foi Agosto (R$ 140.320,55 com 13 vendas), enquanto Janeiro apresentou a menor receita (R$ 47.169,34 com 9 vendas). Outubro liderou em volume com 118 itens vendidos.

2. Desempenho de Produtos: o `Top 5 produtos` é liderado por Tablet (R$ 353.176,81) e Notebook (R$ 350.212,88), que juntos concentram a maior parte do faturamento, seguidos por Smartphone, Monitor e Headset.

3. Visão de Portfólio: a categoria de `Celulares` gerou a maior receita (R$ 588.153,99), seguida de perto por `Computadores` (R$ 540.679,29), enquanto `Periféricos` teve a menor participação (R$ 105.099,64).

4. Análise Regional: a região `Sudeste` lidera tanto em faturamento total (R$ 329.458,34) quanto em ticket médio por venda (R$ 8.447,65). As demais regiões apresentam receitas equilibradas, variando entre R$ 212 mil e R$ 244 mil.

In [ ]:
def segmentar_clientes(df_limpo): 
    """Segmenta clientes pelo total gasto usando groupby e lambda.""" 
    clientes = df_limpo.groupby("cliente")["receita_total"].sum().reset_index() 
    clientes.columns = ["cliente", "total_gasto"] 

    # Classificação usando função lambda com condicionais 
    clientes["segmento"] = clientes["total_gasto"].apply( 
        lambda gasto: "Ouro" if gasto > 15000 
        else ("Prata" if gasto >= 5000 else "Bronze") 
        ) 
    clientes = clientes.sort_values("total_gasto", ascending=False)

    print("\n== SEGMENTAÇÃO DE CLIENTES ==") 
    print(clientes.head(10).to_string(index=False)) 
    print(f"\nDistribuição de segmentos:\n{clientes['segmento'].value_counts()}") 

    return clientes

In [ ]:
# Chamando a função segmentar_clientes
df_segmentado = segmentar_clientes(df_transformado)

<u>Segmentação de Clientes por Nível de Gasto:</u>

1. Perfil de Consumo Elevado: o `Cliente_015` lidera o ranking de gastos com um faturamento acumulado de R$ 82.964,76, seguido de perto pelo `Cliente_035` com R$ 75.402,52.
2. Concentração no Topo: a maior parte da base de clientes ativa se concentra na faixa de maior poder aquisitivo, com 30 clientes classificados no segmento `Ouro` (gastos acima de R$ 15.000).
3. Distribuição da Base: o restante dos clientes está distribuído em menor proporção nas categorias intermediárias e de entrada, sendo 11 clientes no segmento `Prata` e 8 clientes no segmento `Bronze`.
4. Prontidão Preditiva: essa segmentação fornece uma excelente base para criar modelos preditivos de classificação (ex: prever quais novos clientes se tornarão categoria Ouro) ou campanhas de marketing direcionadas.

### Cálculos estatísticos

In [ ]:
def calcular_estatisticas_numpy(df_limpo): 
    """Usa NumPy para calcular estatísticas sobre as receitas.""" 
    print("\n=== ESTATÍSTICAS COM NUMPY ===") 
 
    receitas = df_limpo["receita_total"].to_numpy()  # Converte para array NumPy 
 
    media = np.mean(receitas) 
    mediana = np.median(receitas) 
    desvio_padrao = np.std(receitas) 
    total = np.sum(receitas) 
    p25 = np.percentile(receitas, 25) 
    p75 = np.percentile(receitas, 75) 
 
    print(f"  Receita média por venda:    R$ {media:.2f}") 
    print(f"  Receita mediana por venda:  R$ {mediana:.2f}") 
    print(f"  Desvio padrão:              R$ {desvio_padrao:.2f}") 
    print(f"  Receita total:              R$ {total:.2f}") 
    print(f"  Percentil 25 (Q1):          R$ {p25:.2f}") 
    print(f"  Percentil 75 (Q3):          R$ {p75:.2f}") 
 
    # Broadcasting: normalizar receitas entre 0 e 1 
    receitas_normalizadas = (receitas - receitas.min()) / (receitas.max() - receitas.min()) 
    print(f"\n  Receitas normalizadas (primeiros 5): {receitas_normalizadas[:5].round(4)}") 
 
    # Operação vetorizada: identificar vendas acima da média sem loop 
    acima_da_media = receitas[receitas > media] 
    print(f"\n  Vendas acima da média: {len(acima_da_media)} de {len(receitas)}") 
 
    return { 
        "media": media, "mediana": mediana, 
        "desvio_padrao": desvio_padrao, "total": total 
    }

In [ ]:
# Executando a função com o DataFrame correto
estatisticas = calcular_estatisticas_numpy(df_transformado)

<u>Cálculos Estatísticos (Análise Descritiva com NumPy):</u>

1. Dispersão e Assimetria: a receita média por venda é de R$ 6.855,18, enquanto a mediana fica em R$ 3.899,61. Essa diferença expressiva indica uma assimetria à direita, causada por vendas de valores muito altos que puxam a média para cima.
2. Volatilidade dos Dados: o desvio padrão de R$ 6.673,94 é muito próximo do valor da média, confirmando uma alta variabilidade e dispersão nos valores dos pedidos do dataset.
3. Distribuição em Quartis: 25% das vendas geram no máximo R$ 1.267,56 (Q1), enquanto os 25% superiores (Q3) representam transações acima de R$ 10.976,68. O faturamento acumulado totalizou R$ 1.233.932,92.
4. Vetorização e Normalização: através de máscaras booleanas, identificou-se que 74 das 180 vendas (41,1%) estão acima da média geral. As receitas também foram normalizadas via broadcasting em uma escala de 0 a 1 para prepará-las para futuros algoritmos de Machine Learning.

### Visualizações gráficas 

In [ ]:
def gerar_visualizacoes(df_transformado, metricas_finais, output_dir="outputs/graficos"):
    """Gera e exporta visualizações dos dados de vendas.""" 
    os.makedirs(output_dir, exist_ok=True)

    # Configurações visuais globais 
    sns.set_theme(style="whitegrid", palette="muted")
    plt.rcParams["figure.figsize"] = (12, 6)
    plt.rcParams["axes.titlesize"] = 14
    plt.rcParams["axes.labelsize"] = 12

    # --- Gráfico 1: Receita por Mês (linha) --- 
    fig, ax = plt.subplots() 
    por_mes = metricas_finais["receita por mes"] 
    ax.plot(por_mes["mes"], por_mes["receita_total"], marker="o", 
    linewidth=2, color="#2196F3") 
    ax.fill_between(por_mes["mes"], por_mes["receita_total"], alpha=0.15, 
    color="#2196F3") 
    ax.set_title("Receita Total por Mês (2024)") 
    ax.set_xlabel("Mês") 
    ax.set_ylabel("Receita Total (R$)") 
    ax.set_xticks(range(1, 13)) 
    ax.set_xticklabels(["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"],rotation=45) 
    plt.tight_layout() 
    caminho = os.path.join(output_dir, "receita_por_mes.png") 
    plt.savefig(caminho, dpi=150) 
    plt.close() 
    print(f"  Gráfico exportado: {caminho}")

    # --- Gráfico 2: Top 5 Produtos (barras horizontais) --- 
    fig, ax = plt.subplots() 
    top = metricas_finais["top 5 produtos por receita"] 
    sns.barplot(data=top, y="produto", x="receita_total", hue="produto", ax=ax, palette="Blues_d", legend=False)
    ax.set_title("Top 5 Produtos por Receita Total") 
    ax.set_xlabel("Receita Total (R$)") 
    ax.set_ylabel("Produto") 
    for container in ax.containers: 
        ax.bar_label(container, fmt="R$ %.0f", padding=5) 
    plt.tight_layout() 
    caminho = os.path.join(output_dir, "top_produtos.png") 
    plt.savefig(caminho, dpi=150) 
    plt.close() 
    print(f"  Gráfico exportado: {caminho}") 
 
    # --- Gráfico 3: Distribuição de Receita por Região (boxplot) --- 
    fig, ax = plt.subplots() 
    sns.boxplot(data=df_transformado, x="regiao", y="receita_total", hue="regiao", ax=ax, palette="Set2") 
    ax.set_title("Distribuição de Receita por Transação – Por Região") 
    ax.set_xlabel("Região") 
    ax.set_ylabel("Receita por Venda (R$)") 
    plt.xticks(rotation=30) 
    plt.tight_layout() 
    caminho = os.path.join(output_dir, "distribuicao_regioes.png") 
    plt.savefig(caminho, dpi=150) 
    plt.close() 
    print(f"  Gráfico exportado: {caminho}") 
 
    print("\n=== VISUALIZAÇÕES GERADAS COM SUCESSO ===")

In [ ]:
# Visualização dos gráficos com os dados tratados e as métricas calculadas

gerar_visualizacoes(df_transformado, metricas_finais)

### Encapsulamento do pipeline de análise de vendas

In [ ]:
class AnalisadorDeVendas: 
    """ 
    Classe responsável por encapsular o pipeline de análise de vendas. 
    Mantém o estado do DataFrame e os resultados intermediários. 
    """ 
 
    def __init__(self, caminho_arquivo): 
        """Inicializa o analisador com o caminho do arquivo de dados.""" 
        self.caminho_arquivo = caminho_arquivo 
        self.df_bruto = None 
        self.df_limpo = None 
        self.metricas = {} 
        self.clientes = None 
        self.relatorio_limpeza = {} 
 
    def carregar(self): 
        """Lê o arquivo CSV e armazena o DataFrame bruto.""" 
        self.df_bruto = pd.read_csv(self.caminho_arquivo, sep=";") 
        print(f"[AnalisadorDeVendas] Arquivo carregado: {self.caminho_arquivo}") 
        print(f"  Registros carregados: {len(self.df_bruto)}")

        return self 
 
    def limpar(self): 
        """Limpa os dados e armazena o DataFrame tratado.""" 
        self.df_limpo, self.relatorio_limpeza = limpar_dados(self.df_bruto.copy())

        return self 
 
    def transformar(self): 
        """Aplica transformações e cria colunas derivadas.""" 
        self.df_limpo = criar_colunas_derivadas(self.df_limpo)

        return self 
 
    def analisar(self): 
        """Calcula métricas e segmentações.""" 
        self.metricas = calcular_metricas(self.df_limpo) 
        self.clientes = segmentar_clientes(self.df_limpo) 
        calcular_estatisticas_numpy(self.df_limpo)

        return self 
 
    def visualizar(self): 
        """Gera e exporta os gráficos.""" 
        gerar_visualizacoes(self.df_limpo, self.metricas)

        return self 
 
    def exportar_relatorio(self, caminho="outputs/relatorio_resumo.csv"): 
        """Exporta o relatório de métricas por mês em CSV e JSON.""" 
        os.makedirs("outputs", exist_ok=True)

        # Exportar CSV
        self.metricas["receita por mes"].to_csv(caminho, index=False)
        print(f"\n[AnalisadorDeVendas] Relatório exportado: {caminho}")

        # Exportar JSON
        caminho_json = "outputs/relatorio_resumo.json"
        with open(caminho_json, "w", encoding="utf-8") as f:
            json.dump(
                self.metricas["receita por mes"].to_dict(orient="records"),
                f,indent=4,ensure_ascii=False
            )

        print(f"[AnalisadorDeVendas] Relatório exportado: {caminho_json}")

        # Ler o JSON exportado para confirmar que o arquivo foi gravado corretamente
        with open(caminho_json, "r", encoding="utf-8") as f:
            dados_lidos = json.load(f)

        print(f"[AnalisadorDeVendas] JSON lido para conferência: {len(dados_lidos)} registros")

        return self
 
    def resumo(self): 
        """Exibe um resumo executivo do pipeline.""" 
        print("\n" + "="*50) 
        print("       RESUMO EXECUTIVO – SALESINSIGHT PY") 
        print("="*50) 
        print(f"  Arquivo analisado:      {self.caminho_arquivo}") 
        print(f"  Registros brutos:       {self.relatorio_limpeza.get('registros_iniciais', 'N/A')}") 
        print(f"  Registros limpos:       {self.relatorio_limpeza.get('registros_finais', 'N/A')}") 
        receita = self.df_limpo["receita_total"].sum() if self.df_limpo is not None else 0 
        print(f"  Receita total anual:    R$ {receita:,.2f}") 
        if self.clientes is not None: 
            top = self.clientes.iloc[0] 
            print(f"  Cliente top:            {top['cliente']} (R$ {top['total_gasto']:,.2f})") 
        print("="*50)

In [ ]:
# --- CHAMADA DO PIPELINE COMPLETO (ENCADEADO) ---

pipeline = AnalisadorDeVendas("./vendas.csv")
pipeline.carregar().limpar().transformar().analisar().visualizar().exportar_relatorio().resumo()

<u>Resumo do Pipeline (SalesInsight Py):</u>

1. Processamento Ponta a Ponta: Consolidação bem-sucedida de todas as etapas analíticas sob uma única interface fluente orientada a objetos.
2. Resultados de Negócio:
   * Base Consolidada: 180 registros higienizados a partir de 200 iniciais (retenção de 90%).
   * Faturamento Anual Acumulado: R$ 1.233.932,92 gerados em 2024.
   * Liderança de Consumo: `Cliente_015` estabelecido como o principal comprador da base, com um gasto total de R$ 82.964,76.
3. Persistência de Saídas: Relatório temporal com 12 registros mensais exportado e validado nativamente nos formatos estruturados `CSV` e `JSON`.

### Projeções

In [ ]:
class AnalisadorComProjecao(AnalisadorDeVendas): 
    """ 
    Extensão do AnalisadorDeVendas com funcionalidades de projeção simples. 
    Herda todos os métodos da classe pai e adiciona projeção de tendência. 
    """ 
 
    def __init__(self, caminho_arquivo, meses_projecao=3): 
        super().__init__(caminho_arquivo) 
        self.meses_projecao = meses_projecao 
        self.projecoes = [] 
 
    def projetar_tendencia(self): 
        """ 
        Projeta a receita dos próximos meses com base na média móvel dos últimos 3 meses. 
        Método simples sem machine learning – baseado em médias. 
        """ 
        if not self.metricas or "receita por mes" not in self.metricas: 
            print("[AVISO] Rode .analisar() antes de projetar.") 
            return self 
 
        por_mes = self.metricas["receita por mes"].sort_values("mes") 
        receitas_historicas = por_mes["receita_total"].to_numpy() 
 
        # Média móvel dos últimos 3 meses como base da projeção 
        ultimos_3 = receitas_historicas[-3:] 
        media_movel = np.mean(ultimos_3) 
        tendencia = np.std(ultimos_3) * 0.1  # fator de crescimento simples 
 
        ultimo_mes = int(por_mes["mes"].max()) 
 
        print("\n=== PROJEÇÃO DE TENDÊNCIA (Média Móvel Simples) ===")

        print(f"  Base: média dos últimos 3 meses = R$ {media_movel:,.2f}") 
        self.projecoes = [] 
 
        for i in range(1, self.meses_projecao + 1): 
            mes_projetado = (ultimo_mes + i - 1) % 12 + 1 
            receita_projetada = media_movel + (tendencia * i) 
            self.projecoes.append({"mes": mes_projetado, "receita_projetada": round(receita_projetada, 2)}) 
            print(f"  Mês {mes_projetado:02d} (projeção): R$ {receita_projetada:,.2f}") 
 
        return self 
 
    def exibir_projecao_detalhada(self): 
        """Exibe as projeções calculadas.""" 
        if not self.projecoes: 
            print("[AVISO] Nenhuma projeção disponível. Rode .projetar_tendencia() primeiro.") 
            return 
        print("\n=== DETALHAMENTO DAS PROJEÇÕES ===") 
        for p in self.projecoes: 
            print(f"  Mês {p['mes']:02d}: R$ {p['receita_projetada']:,.2f}")

In [ ]:
# --- NOVA CHAMADA DO PIPELINE USANDO HERANÇA E PROJEÇÃO ---
# Criamos a instância a partir da classe filha (AnalisadorComProjecao)

pipeline_projecao = AnalisadorComProjecao("./vendas.csv", meses_projecao=3)

# Executamos toda a cadeia fluente incluindo o novo método de projetar
(
    pipeline_projecao.carregar()
    .limpar()
    .transformar()
    .analisar()
    .visualizar()
    .exportar_relatorio()
    .projetar_tendencia()  # <- Novo método da classe filha
    .resumo()
)

<u>Projeções de Faturamento para o Próximo Trimestre:</u>

* **Mês 01 (Janeiro):** R$ 110.432,89 (Projeção)
* **Mês 02 (Fevereiro):** R$ 113.255,56 (Projeção)
* **Mês 03 (Março):** R$ 116,078,23 (Projeção)

*Nota: As estimativas foram calculadas de forma automatizada pela classe com base na média móvel de faturamento dos últimos três meses históricos (Outubro, Novembro e Dezembro), aplicando um fator de crescimento linear derivado do desvio padrão.*

### Executando pipeline completo

In [ ]:
def main(): 
    """ 
    Função principal: executa o pipeline completo do SalesInsight PY. 
    """ 
    print("\n" + "="*60) 
    print("   SALESINSIGHT PY – Pipeline de Análise de Dados de Vendas") 
    print("="*60) 
 
    # Etapa 0: Gerar dataset (Dataset gerado com arquivo "gerar_csv.py") 
    
    # Etapa 1 a 6: Pipeline via classe com herança 
    analisador = AnalisadorComProjecao("vendas.csv", meses_projecao=3) 
    (analisador 
        .carregar() 
        .limpar() 
        .transformar() 
        .analisar() 
        .projetar_tendencia() 
        .visualizar() 
        .exportar_relatorio() 
    ) 

    # Resumo final 
    analisador.resumo() 
    analisador.exibir_projecao_detalhada() 
 
    print("\n[CONCLUÍDO] Pipeline finalizado com sucesso!") 
 
 
if __name__ == "__main__": 
    main()

<u>Conclusão do Pipeline:</u>

* Ponto de Entrada (`main`): Centralização de todas as etapas analíticas e preditivas em uma rotina de execução unificada.
* Orquestração de Fluxo: Acionamento seguro e sequencial do pipeline através da interface fluente da classe estendida, garantindo que o carregamento, a higienização, a engenharia de atributos, a modelagem estatística, os gráficos e os arquivos CSV/JSON sejam gerados de forma integrada.
* Status: O pipeline foi concluído com sucesso e o dataset está pronto para auditoria ou futuras implementações de Machine Learning.